# 1-minute scalp signal training

30개의 연속 1분봉으로 `take_profit` 선도달 여부(TP vs 나머지)를 분류한다. 날짜(session) 단위 분할, train-only 정규화, CSV→mmap 캐시, mixed precision, early stopping을 포함한다. 모델은 짧은 시계열에 맞춘 multi-scale TCN + Patch Transformer 하이브리드다.


In [ ]:
from pathlib import Path
import json
import math
import os
import random
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('PyTorch가 없습니다. urban 커널에 torch를 설치한 뒤 다시 실행하세요.') from exc

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

SEED = 42
BATCH_SIZE = 256
EPOCHS = 40
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
PATIENCE = 7
NUM_WORKERS = min(8, os.cpu_count() or 1)
COMPILE_MODEL = False

AI_READY_ROOT = Path('../../data/stock_data/ai_ready').resolve()
LABEL_PATH = AI_READY_ROOT / 'label.csv'
INPUT_ROOT = AI_READY_ROOT / 'input'
CACHE_ROOT = AI_READY_ROOT / 'train_cache'
CHECKPOINT_PATH = '../../model/stock/best_patch_tcn_tp_binary.pt'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.mkdir('../../model/stock') if not os.path.exists('../../model/stock') else None
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('torch:', torch.__version__)
print('device:', DEVICE)
print('data:', AI_READY_ROOT)


In [ ]:
labels = pd.read_csv(LABEL_PATH)
required_label_columns = {'label', 'realized_net_return'}
missing_label_columns = required_label_columns.difference(labels.columns)
if missing_label_columns:
    raise ValueError(
        f'label.csv에 {sorted(missing_label_columns)} 컬럼이 없습니다. '
        'DataPreprocessing.ipynb의 셀 4를 다시 실행하세요.'
    )
labels['entry_timestamp_kst'] = pd.to_datetime(labels['entry_timestamp_kst'], errors='raise')
labels['target'] = labels['label'].eq(1).astype(np.float32)  # TP=1, SL/timeout=0

# 자정 이후까지 이어지는 미국 세션이 쪼개지지 않도록 파일명의 session date로 분할한다.
labels['session_date'] = pd.to_datetime(
    labels['source_file'].str.extract(r'_(\d{4}-\d{2}-\d{2})_', expand=False),
    errors='raise',
).dt.date
labels = labels.sort_values(['entry_timestamp_kst', 'sample_id']).reset_index(drop=True)

session_dates = np.array(sorted(labels['session_date'].unique()))
if len(session_dates) < 5:
    raise ValueError('안전한 날짜 단위 train/val/test 분할에는 최소 5개 세션이 필요합니다.')
n_train_dates = max(1, int(len(session_dates) * 0.70))
n_val_dates = max(1, int(len(session_dates) * 0.15))
train_dates = set(session_dates[:n_train_dates])
val_dates = set(session_dates[n_train_dates:n_train_dates + n_val_dates])
test_dates = set(session_dates[n_train_dates + n_val_dates:])
if not test_dates:
    raise ValueError('test 세션이 비었습니다. 분할 비율을 조정하세요.')

split = np.full(len(labels), 'test', dtype=object)
split[labels['session_date'].isin(train_dates)] = 'train'
split[labels['session_date'].isin(val_dates)] = 'val'
labels['split'] = split

display(pd.crosstab(labels['split'], labels['label'], margins=True))
print('train dates:', sorted(train_dates))
print('val dates  :', sorted(val_dates))
print('test dates :', sorted(test_dates))


In [ ]:
FEATURE_COLUMNS = [
    'open', 'high', 'low', 'close', 'volume', 'trade_strength',
    'aggressive_buy_volume', 'aggressive_sell_volume', 'session_vwap',
    'return_pct', 'bid_ask_imbalance', 'avg_bid_size', 'avg_ask_size',
    'avg_spread_pct', 'ema9', 'ema20', 'volume_ratio_ma20',
    'vwap_gap_ratio', 'ema_momentum', 'aggressive_buy_ratio',
    'capital_inflow_ratio_ma20', 'spread_ratio_ma20',
    'orderbook_imbalance_strength',
]
PRICE_COLUMNS = ['open', 'high', 'low', 'close', 'session_vwap', 'ema9', 'ema20']
LOG1P_COLUMNS = ['volume', 'aggressive_buy_volume', 'aggressive_sell_volume', 'avg_bid_size', 'avg_ask_size']
PERCENT_COLUMNS = ['trade_strength', 'return_pct', 'avg_spread_pct']
MISSING_SENTINEL_COLUMNS = [
    'trade_strength', 'avg_spread_pct', 'avg_bid_size', 'avg_ask_size',
    'bid_ask_imbalance', 'spread_ratio_ma20', 'orderbook_imbalance_strength',
]
MODEL_FEATURES = FEATURE_COLUMNS + [f'{c}__missing' for c in MISSING_SENTINEL_COLUMNS]
SEQUENCE_LENGTH = 30
CACHE_VERSION = 2
CACHE_PATH = CACHE_ROOT / 'features_float32.npy'
CACHE_META_PATH = CACHE_ROOT / 'features_meta.json'

def transform_input_frame(frame):
    if len(frame) != SEQUENCE_LENGTH:
        raise ValueError(f'expected {SEQUENCE_LENGTH} rows, got {len(frame)}')
    values = frame[FEATURE_COLUMNS].astype(np.float64).copy()
    missing_parts = []
    for column in MISSING_SENTINEL_COLUMNS:
        missing = values[column].lt(0) | values[column].isna()
        missing_parts.append(missing.to_numpy(np.float32)[:, None])
        values.loc[missing, column] = np.nan

    last_close = float(values['close'].iloc[-1])
    if not np.isfinite(last_close) or last_close <= 0:
        raise ValueError('invalid last close')
    for column in PRICE_COLUMNS:
        ratio = values[column].clip(lower=1e-8) / last_close
        values[column] = np.log(ratio).clip(-2, 2)
    for column in LOG1P_COLUMNS:
        values[column] = np.log1p(values[column].clip(lower=0))
    for column in PERCENT_COLUMNS:
        values[column] = values[column] / 100.0

    array = values.to_numpy(np.float32)
    array = np.nan_to_num(array, nan=0.0, posinf=20.0, neginf=-20.0)
    array = np.clip(array, -20.0, 20.0)
    return np.concatenate([array, *missing_parts], axis=1).astype(np.float32)

expected_meta = {
    'version': CACHE_VERSION,
    'rows': len(labels),
    'sequence_length': SEQUENCE_LENGTH,
    'features': MODEL_FEATURES,
    'label_mtime_ns': LABEL_PATH.stat().st_mtime_ns,
}
cache_valid = False
if CACHE_PATH.exists() and CACHE_META_PATH.exists():
    cache_valid = json.loads(CACHE_META_PATH.read_text()) == expected_meta

if not cache_valid:
    CACHE_PATH.unlink(missing_ok=True)
    CACHE_META_PATH.unlink(missing_ok=True)
    cache = np.lib.format.open_memmap(
        CACHE_PATH, mode='w+', dtype=np.float32,
        shape=(len(labels), SEQUENCE_LENGTH, len(MODEL_FEATURES)),
    )
    for row_index, relative_path in enumerate(tqdm(labels['input_file'], desc='Build mmap cache')):
        frame = pd.read_csv(AI_READY_ROOT / relative_path, usecols=FEATURE_COLUMNS)
        cache[row_index] = transform_input_frame(frame)
    cache.flush()
    del cache
    CACHE_META_PATH.write_text(json.dumps(expected_meta, indent=2))

features_mmap = np.load(CACHE_PATH, mmap_mode='r')
print('cache:', features_mmap.shape, CACHE_PATH)


In [ ]:
train_indices = np.flatnonzero(labels['split'].eq('train').to_numpy())
val_indices = np.flatnonzero(labels['split'].eq('val').to_numpy())
test_indices = np.flatnonzero(labels['split'].eq('test').to_numpy())
targets = labels['target'].to_numpy(np.float32)

# 정규화 통계는 train 데이터에서만 계산한다.
feature_sum = np.zeros(len(MODEL_FEATURES), dtype=np.float64)
feature_sq_sum = np.zeros(len(MODEL_FEATURES), dtype=np.float64)
feature_count = 0
for start in range(0, len(train_indices), 4096):
    batch = np.asarray(features_mmap[train_indices[start:start + 4096]], dtype=np.float64)
    feature_sum += batch.sum(axis=(0, 1))
    feature_sq_sum += np.square(batch).sum(axis=(0, 1))
    feature_count += batch.shape[0] * batch.shape[1]
feature_mean = feature_sum / feature_count
feature_var = np.maximum(feature_sq_sum / feature_count - feature_mean ** 2, 1e-8)
feature_std = np.sqrt(feature_var)

positive_count = int(targets[train_indices].sum())
negative_count = len(train_indices) - positive_count
positive_weight = math.sqrt(negative_count / max(positive_count, 1))
print({'non_tp': negative_count, 'take_profit': positive_count})
print(f'positive weight: {positive_weight:.3f}')

class StockWindowDataset(Dataset):
    def __init__(self, cache_path, indices, targets, mean, std):
        self.cache_path = cache_path
        self.indices = np.asarray(indices, dtype=np.int64)
        self.targets = targets
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self._cache = None

    def __len__(self):
        return len(self.indices)

    def _get_cache(self):
        if self._cache is None:
            self._cache = np.load(self.cache_path, mmap_mode='r')
        return self._cache

    def __getitem__(self, item):
        index = self.indices[item]
        x = np.array(self._get_cache()[index], dtype=np.float32, copy=True)
        x = np.clip((x - self.mean) / self.std, -8.0, 8.0)
        return torch.from_numpy(x), torch.tensor(self.targets[index], dtype=torch.float32), index

train_dataset = StockWindowDataset(CACHE_PATH, train_indices, targets, feature_mean, feature_std)
val_dataset = StockWindowDataset(CACHE_PATH, val_indices, targets, feature_mean, feature_std)
test_dataset = StockWindowDataset(CACHE_PATH, test_indices, targets, feature_mean, feature_std)

loader_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=AMP_ENABLED,
    persistent_workers=NUM_WORKERS > 0,
)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
print('dataset sizes:', len(train_dataset), len(val_dataset), len(test_dataset))


In [ ]:
class ResidualTCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation, groups=channels),
            nn.Conv1d(channels, channels * 2, 1),
            nn.GLU(dim=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)

class PatchTCNClassifier(nn.Module):
    def __init__(
        self, n_features, sequence_length=30, n_classes=1, d_model=96,
        patch_size=5, patch_stride=2, n_heads=8, n_layers=2, dropout=0.25,
    ):
        super().__init__()
        self.config = dict(
            n_features=n_features, sequence_length=sequence_length, n_classes=n_classes,
            d_model=d_model, patch_size=patch_size, patch_stride=patch_stride,
            n_heads=n_heads, n_layers=n_layers, dropout=dropout,
        )
        self.feature_gate = nn.Parameter(torch.zeros(n_features))
        self.input_norm = nn.LayerNorm(n_features)

        self.local_stem = nn.Conv1d(n_features, d_model, 1)
        self.local_blocks = nn.Sequential(
            ResidualTCNBlock(d_model, 3, 1, dropout),
            ResidualTCNBlock(d_model, 3, 2, dropout),
            ResidualTCNBlock(d_model, 3, 4, dropout),
        )
        self.local_attention = nn.Conv1d(d_model, 1, 1)

        self.patch_embed = nn.Conv1d(n_features, d_model, patch_size, stride=patch_stride)
        n_patches = (sequence_length - patch_size) // patch_stride + 1
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.position = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model))
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, x):
        x = self.input_norm(x)
        x = x * (2.0 * torch.sigmoid(self.feature_gate))[None, None, :]
        channel_first = x.transpose(1, 2)

        local = self.local_blocks(self.local_stem(channel_first))
        local_weights = torch.softmax(self.local_attention(local), dim=-1)
        local_embedding = (local * local_weights).sum(dim=-1)

        patches = self.patch_embed(channel_first).transpose(1, 2)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, patches], dim=1)
        tokens = self.transformer(tokens + self.position[:, :tokens.size(1)])
        global_embedding = tokens[:, 0]
        return self.head(torch.cat([local_embedding, global_embedding], dim=-1)).squeeze(-1)

model = PatchTCNClassifier(n_features=len(MODEL_FEATURES)).to(DEVICE)
if COMPILE_MODEL and hasattr(torch, 'compile'):
    model = torch.compile(model)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'trainable parameters: {trainable_parameters:,}')


In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(positive_weight, dtype=torch.float32, device=DEVICE),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE, epochs=EPOCHS, steps_per_epoch=len(train_loader),
    pct_start=0.15, div_factor=10.0, final_div_factor=100.0,
)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = probabilities >= threshold
    true_positive = y_true == 1
    return {
        'f1': f1_score(y_true, predictions, zero_division=0),
        'balanced_acc': balanced_accuracy_score(y_true, predictions),
        'tp_precision': (
            (predictions & true_positive).sum() / max(predictions.sum(), 1)
        ),
        'tp_recall': (
            (predictions & true_positive).sum() / max(true_positive.sum(), 1)
        ),
        'tp_pr_auc': average_precision_score(true_positive.astype(int), probabilities),
    }

def run_epoch(loader, training):
    model.train(training)
    total_loss, seen = 0.0, 0
    all_probabilities, all_targets, all_indices = [], [], []
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for x, y, indices in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            seen += batch_size
            all_probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())
            all_indices.append(indices.numpy())
    probabilities = np.concatenate(all_probabilities)
    y_true = np.concatenate(all_targets)
    metrics = classification_metrics(y_true, probabilities)
    metrics['loss'] = total_loss / seen
    return metrics, y_true, probabilities, np.concatenate(all_indices)


In [ ]:
history = []
best_score = -np.inf
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_metrics, *_ = run_epoch(train_loader, training=True)
    val_metrics, *_ = run_epoch(val_loader, training=False)
    record = {
        'epoch': epoch,
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'seconds': time.time() - started,
    }
    history.append(record)
    print(
        f"epoch {epoch:02d} | train loss {train_metrics['loss']:.4f} "
        f"| val loss {val_metrics['loss']:.4f} | val F1 {val_metrics['f1']:.4f} "
        f"| TP P/R {val_metrics['tp_precision']:.3f}/{val_metrics['tp_recall']:.3f} "
        f"| PR-AUC {val_metrics['tp_pr_auc']:.4f} | {record['seconds']:.1f}s"
    )

    score = val_metrics['tp_pr_auc']
    if score > best_score:
        best_score = score
        epochs_without_improvement = 0
        state_dict = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
        torch.save({
            'model_state': state_dict,
            'model_config': model._orig_mod.config if hasattr(model, '_orig_mod') else model.config,
            'feature_columns': MODEL_FEATURES,
            'feature_mean': feature_mean.astype(np.float32),
            'feature_std': feature_std.astype(np.float32),
            'target_definition': 'take_profit_vs_rest',
            'epoch': epoch,
            'val_metrics': val_metrics,
        }, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'early stopping at epoch {epoch}')
            break

history_df = pd.DataFrame(history)
display(history_df.tail())


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
feature_mean = np.asarray(checkpoint['feature_mean'], dtype=np.float32)
feature_std = np.asarray(checkpoint['feature_std'], dtype=np.float32)
best_model = PatchTCNClassifier(**checkpoint['model_config']).to(DEVICE)
best_model.load_state_dict(checkpoint['model_state'])
model = best_model

val_metrics, val_y, val_prob, val_row_indices = run_epoch(val_loader, training=False)
test_metrics, test_y, test_prob, test_row_indices = run_epoch(test_loader, training=False)
print('test metrics:', {k: round(v, 4) for k, v in test_metrics.items()})
test_prediction = test_prob >= 0.5
print(classification_report(
    test_y, test_prediction, target_names=['non_tp', 'take_profit'], digits=4, zero_division=0,
))
display(pd.DataFrame(
    confusion_matrix(test_y, test_prediction),
    index=['true_non_tp', 'true_take_profit'],
    columns=['pred_non_tp', 'pred_take_profit'],
))

def trading_metrics(probabilities, y_true, row_indices, threshold):
    selected = probabilities >= threshold
    signal_count = int(selected.sum())
    true_positive = y_true == 1
    precision = (selected & true_positive).sum() / max(signal_count, 1)
    recall = (selected & true_positive).sum() / max(true_positive.sum(), 1)
    returns = labels.iloc[row_indices]['realized_net_return'].to_numpy(np.float64)[selected]
    if signal_count == 0:
        return dict(threshold=threshold, signals=0, tp_precision=0.0, tp_recall=0.0,
                    mean_net_return=np.nan, compounded_return=np.nan, profit_factor=np.nan, max_drawdown=np.nan)
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum()
    equity = np.cumprod(1 + np.clip(returns, -0.999, None))
    running_peak = np.maximum.accumulate(np.r_[1.0, equity])
    drawdown = np.r_[1.0, equity] / running_peak - 1
    return dict(
        threshold=float(threshold), signals=signal_count, tp_precision=float(precision),
        tp_recall=float(recall), mean_net_return=float(returns.mean()),
        compounded_return=float(equity[-1] - 1),
        profit_factor=float(gains / losses) if losses > 0 else np.inf,
        max_drawdown=float(drawdown.min()),
    )

# Threshold는 validation 실제 순수익으로만 선택하고 test에는 고정 적용한다.
threshold_table = pd.DataFrame([
    trading_metrics(val_prob, val_y, val_row_indices, threshold)
    for threshold in np.arange(0.10, 0.91, 0.02)
])
MIN_VAL_SIGNALS = 100
threshold_candidates = threshold_table[threshold_table['signals'] >= MIN_VAL_SIGNALS].copy()
if threshold_candidates.empty:
    raise ValueError('최소 validation 신호 수를 만족하는 threshold가 없습니다.')
best_threshold = float(
    threshold_candidates.loc[threshold_candidates['mean_net_return'].idxmax(), 'threshold']
)
display(threshold_table.sort_values('mean_net_return', ascending=False).head(15))
test_trading_result = trading_metrics(test_prob, test_y, test_row_indices, best_threshold)
print(f'selected validation threshold: {best_threshold:.2f}')
display(pd.Series(test_trading_result, name='test'))


In [ ]:
# Validation permutation importance
# 한 컬럼의 30봉 전체 궤적을 샘플끼리 섞어서 TP PR-AUC 하락량을 측정한다.
PERMUTATION_REPEATS = 2
IMPORTANCE_BATCH_SIZE = 512

def predict_probability_array(model, array, batch_size=IMPORTANCE_BATCH_SIZE):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(array), batch_size):
            x = torch.from_numpy(array[start:start + batch_size]).to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(x)
            outputs.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(outputs)

# mmap 원본을 validation 크기만큼만 메모리에 올리고 train 통계로 정규화한다.
importance_x = np.asarray(features_mmap[val_indices], dtype=np.float32).copy()
importance_x = np.clip(
    (importance_x - feature_mean[None, None, :]) / feature_std[None, None, :],
    -8.0, 8.0,
).astype(np.float32)
importance_y = targets[val_indices]
baseline_probability = predict_probability_array(model, importance_x)
baseline_pr_auc = average_precision_score(
    importance_y.astype(np.int8), baseline_probability,
)
baseline_macro_f1 = f1_score(
    importance_y, baseline_probability >= 0.5, zero_division=0,
)

rng = np.random.default_rng(SEED)
importance_records = []
for feature_index, feature_name in enumerate(tqdm(MODEL_FEATURES, desc='Permutation importance')):
    pr_auc_drops = []
    macro_f1_drops = []
    for _ in range(PERMUTATION_REPEATS):
        permutation = rng.permutation(len(importance_x))
        original_feature = importance_x[:, :, feature_index].copy()
        importance_x[:, :, feature_index] = original_feature[permutation]
        permuted_probability = predict_probability_array(model, importance_x)
        permuted_pr_auc = average_precision_score(
            importance_y.astype(np.int8), permuted_probability,
        )
        permuted_macro_f1 = f1_score(
            importance_y, permuted_probability >= 0.5, zero_division=0,
        )
        pr_auc_drops.append(baseline_pr_auc - permuted_pr_auc)
        macro_f1_drops.append(baseline_macro_f1 - permuted_macro_f1)
        importance_x[:, :, feature_index] = original_feature

    importance_records.append({
        'feature': feature_name,
        'tp_pr_auc_drop': float(np.mean(pr_auc_drops)),
        'tp_pr_auc_drop_std': float(np.std(pr_auc_drops)),
        'macro_f1_drop': float(np.mean(macro_f1_drops)),
    })

feature_importance = pd.DataFrame(importance_records).rename(columns={
    'tp_pr_auc_drop': 'val_tp_pr_auc_drop',
    'tp_pr_auc_drop_std': 'val_tp_pr_auc_drop_std',
    'macro_f1_drop': 'val_macro_f1_drop',
})

# 같은 크기의 train 표본에서도 importance를 계산해 일반화되지 않는 컬럼을 찾는다.
train_sample_size = min(len(train_indices), len(val_indices), 10_000)
train_sample_indices = rng.choice(train_indices, size=train_sample_size, replace=False)
train_importance_x = np.asarray(features_mmap[train_sample_indices], dtype=np.float32).copy()
train_importance_x = np.clip(
    (train_importance_x - feature_mean[None, None, :]) / feature_std[None, None, :],
    -8.0, 8.0,
).astype(np.float32)
train_importance_y = targets[train_sample_indices]
train_baseline_probability = predict_probability_array(model, train_importance_x)
train_baseline_pr_auc = average_precision_score(
    train_importance_y.astype(np.int8), train_baseline_probability,
)
train_drop_records = []
for feature_index, feature_name in enumerate(tqdm(MODEL_FEATURES, desc='Train importance')):
    drops = []
    for _ in range(PERMUTATION_REPEATS):
        permutation = rng.permutation(len(train_importance_x))
        original_feature = train_importance_x[:, :, feature_index].copy()
        train_importance_x[:, :, feature_index] = original_feature[permutation]
        probability = predict_probability_array(model, train_importance_x)
        permuted_pr_auc = average_precision_score(
            train_importance_y.astype(np.int8), probability,
        )
        drops.append(train_baseline_pr_auc - permuted_pr_auc)
        train_importance_x[:, :, feature_index] = original_feature
    train_drop_records.append({
        'feature': feature_name,
        'train_tp_pr_auc_drop': float(np.mean(drops)),
    })

feature_importance = feature_importance.merge(pd.DataFrame(train_drop_records), on='feature')
feature_importance['overfit_gap'] = (
    feature_importance['train_tp_pr_auc_drop'] - feature_importance['val_tp_pr_auc_drop']
)
feature_importance['overfit_suspect'] = (
    (feature_importance['train_tp_pr_auc_drop'] > 0)
    & (feature_importance['val_tp_pr_auc_drop'] <= 0)
)
feature_importance = feature_importance.sort_values(
    ['overfit_suspect', 'overfit_gap'], ascending=[False, False],
).reset_index(drop=True)
print(f'train/validation baseline TP PR-AUC: {train_baseline_pr_auc:.4f} / {baseline_pr_auc:.4f}')
display(feature_importance.head(20).style.format({
    'train_tp_pr_auc_drop': '{:.5f}',
    'val_tp_pr_auc_drop': '{:.5f}',
    'val_tp_pr_auc_drop_std': '{:.5f}',
    'val_macro_f1_drop': '{:.5f}',
    'overfit_gap': '{:.5f}',
}))

# 내부 gate는 참고값이다. 큰 gate가 높은 permutation importance를 보장하지 않는다.
gate_values = (2.0 * torch.sigmoid(model.feature_gate)).detach().cpu().numpy()
gate_table = pd.DataFrame({'feature': MODEL_FEATURES, 'gate_multiplier': gate_values})
display(gate_table.sort_values('gate_multiplier', ascending=False).head(20))

feature_importance.to_csv(CACHE_ROOT / 'validation_feature_importance.csv', index=False)
del importance_x, train_importance_x


In [ ]:
def predict_input_csv(csv_path, threshold=best_threshold):
    frame = pd.read_csv(csv_path, usecols=FEATURE_COLUMNS)
    x = transform_input_frame(frame)
    x = np.clip((x - feature_mean) / feature_std, -8.0, 8.0)
    tensor = torch.from_numpy(x.astype(np.float32))[None].to(DEVICE)
    model.eval()
    with torch.inference_mode():
        probability = torch.sigmoid(model(tensor))[0].item()
    return {
        'take_profit_probability': float(probability),
        'buy_signal': bool(probability >= threshold),
        'threshold': float(threshold),
    }

example_path = AI_READY_ROOT / labels.iloc[test_indices[0]]['input_file']
predict_input_csv(example_path)
